### Download and unzip coco 2017 validation dataset

In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip

!unzip val2017.zip
!unzip annotations_trainval2017.zip

### Pair images with their first captions

In [ ]:
import os
import json

COCO_IMG_DIR = "val2017"
COCO_ANN_FILE = "annotations/captions_val2017.json"


def load_coco_val2017():
    with open(COCO_ANN_FILE, "r") as f:
        data = json.load(f)

    # Map image_id -> captions
    captions_dict = {}
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        captions_dict.setdefault(img_id, []).append(ann["caption"])

    # Map image_id -> filename
    id_to_file = {img["id"]: img["file_name"] for img in data["images"]}

    samples = []
    for img_id, file_name in id_to_file.items():
        path = os.path.join(COCO_IMG_DIR, file_name)
        captions = captions_dict.get(img_id, [])

        if os.path.exists(path) and captions:
            samples.append({
                "image_path": path,
                "caption": captions[0]  # pick first caption
            })

    return samples


samples = load_coco_val2017()
print(f"Loaded {len(samples)} samples")

### Save the first 1000 prompts to use for inference later

In [ ]:
import json

prompts = [s["caption"] for s in samples[:1000]]

file_path = "/content/drive/MyDrive/data/prompts.json"

with open(file_path, "w") as f:
    json.dump(prompts, f)

### Resize and save real images

In [ ]:
os.makedirs("coco_real", exist_ok=True)

for i, sample in enumerate(samples):
    img = Image.open(sample["image_path"]).convert("RGB")
    img = img.resize((512, 512))
    img.save(f"coco_real/{i}.png")

print("Saved resized COCO images")

### Get prompts and real images from google drive

In [ ]:
import json

from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/data/prompts.json"

with open(file_path, "r") as f:
    prompts = json.load(f)

!unzip /content/drive/MyDrive/data/coco_real.zip -d /

### Define load pipeline, to use either SD1.5 or SD2.1

In [ ]:
from diffusers import StableDiffusionPipeline, DiffusionPipeline, DDIMScheduler
import torch

def load_pipeline(name):
    if name == "sd1.5":
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16
        )
    elif name == "sd2.1":
        pipe = DiffusionPipeline.from_pretrained(
            "sd2-community/stable-diffusion-2-1",
            torch_dtype=torch.float16
        )

    pipe = pipe.to("cuda")
    return pipe

### Define function for generating images

In [8]:
import os
import time
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image

def generate_images(exp_name, pipe, prompts, out_dir, use_ddim, batch_size=32, num_steps=25):
  '''
  Generate images with given prompts and pipe.
  If use_ddim is True, use DDIMSchedular.
  Save them in the out_dir.
  '''
    os.makedirs(out_dir, exist_ok=True)

    if use_ddim:
        pipe.scheduler = DDIMScheduler.from_config(
            pipe.scheduler.config,
            clip_sample=False,
            set_alpha_to_one=False
        )

    pipe = pipe.to("cuda")

    all_times = []
    all_vram_peaks = []

    torch.cuda.reset_peak_memory_stats()

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i+batch_size]

        generator = torch.Generator(device="cuda").manual_seed(42 + i)

        torch.cuda.synchronize()
        start = time.time()

        images = pipe(
            batch_prompts,
            generator=generator,
            num_inference_steps=num_steps,
            eta=0.0
        ).images

        torch.cuda.synchronize()
        end = time.time()

        all_times.append(end - start)

        for j, img in enumerate(images):
            img.save(f"{out_dir}/{i+j}.png")

    # get peak VRAM
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    # get speed stats
    total_time = sum(all_times)
    total_images = len(prompts)

    avg_time_per_image = total_time / total_images
    images_per_second = total_images / total_time
    std_per_image = np.std([
        t / len(prompts[i:i+batch_size])
        for i, t in zip(range(0, len(prompts), batch_size), all_times)
    ])

    # return time and memory metrics
    return {
        "exp_name": exp_name,
        "total_time": total_time,
        "throughput": images_per_second,
        "avg_time_per_image": avg_time_per_image,
        "std_per_image": std_per_image,
        "peak_vram_mb": peak_vram_mb
    }

### Generate 1000 images with sd1.5

In [ ]:
pipe = load_pipeline("sd1.5")
generate_images("base_sd1.5", pipe, prompts, "base_sd1.5_images", False)

### Save base_sd1.5 images to google drive

In [ ]:
!zip -r base_sd1.5_images.zip /content/base_sd1.5_images
!mv base_sd1.5_images.zip /content/drive/MyDrive/data

### Generate 1000 images with sd1.5 using DDIM

In [ ]:
pipe = load_pipeline("sd1.5")
generate_images("DDIM_sd1.5", pipe, prompts, "DDIM_sd1.5_images", True)

### Save DDIM_sd1.5 images to google drive

In [ ]:
!zip -r DDIM_sd1.5_images.zip /content/DDIM_sd1.5_images
!mv DDIM_sd1.5_images.zip /content/drive/MyDrive/data

### Generate 1000 images with sd1.5 using DDIM and decreasing to 15 inference steps

In [ ]:
pipe = load_pipeline("sd1.5")
generate_images("DDIM_15s_sd1.5", pipe, prompts, "DDIM_15s_sd1.5_images", True, num_steps=15)

### Save DDIM_15s_sd1.5 images to google drive

In [ ]:
!zip -r DDIM_15s_sd1.5_images.zip /content/DDIM_15s_sd1.5_images
!mv DDIM_15s_sd1.5_images.zip /content/drive/MyDrive/data

### Generate 1000 images with sd2.1

In [ ]:
pipe = load_pipeline("sd2.1")
generate_images("base_sd2.1", pipe, prompts, "base_sd2.1_images", False)

### Save base_sd2.1 images to google drive

In [ ]:
!zip -r base_sd2.1_images.zip /content/base_sd2.1_images
!mv base_sd2.1_images.zip /content/drive/MyDrive/data

### Generate 1000 images with sd2.1 using DDIM

In [ ]:
pipe = load_pipeline("sd2.1")
generate_images("DDIM_sd2.1", pipe, prompts, "DDIM_sd2.1_images", True)

### Save DDIM_sd2.1 images to google drive

In [ ]:
!zip -r DDIM_sd2.1_images.zip /content/DDIM_sd2.1_images
!mv DDIM_sd2.1_images.zip /content/drive/MyDrive/data

### Generate 1000 images with sd2.1 using DDIM and decreasing to 15 inference steps

In [ ]:
pipe = load_pipeline("sd2.1")
generate_images("DDIM_15s_sd2.1", pipe, prompts, "DDIM_15s_sd2.1_images", True, num_steps=15)

### Save DDIM_15s_sd2.1 images to google drive

In [ ]:
!zip -r DDIM_15s_sd2.1_images.zip /content/DDIM_15s_sd2.1_images
!mv DDIM_15s_sd2.1_images.zip /content/drive/MyDrive/data

### Install torch_fidelity

In [ ]:
!pip install torch_fidelity

### Define functions for calculating FID and CLIP

In [10]:
import torch.nn.functional as F
from torch_fidelity import calculate_metrics
from transformers import CLIPProcessor, CLIPModel

def compute_fid(real_dir, fake_dir):
    metrics = calculate_metrics(
        input1=real_dir,
        input2=fake_dir,
        fid=True,
        isc=False,
        kid=False,
        cuda=True
    )
    return metrics["frechet_inception_distance"]

def load_clip():
    model = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32"
    ).to("cuda")

    processor = CLIPProcessor.from_pretrained(
        "openai/clip-vit-base-patch32"
    )

    model.eval()
    return model, processor

def compute_clip_score(image_dir, prompts, model, processor, batch_size):
    image_files = sorted(os.listdir(image_dir))

    scores = []

    for i in tqdm(range(0, len(image_files), batch_size)):
        batch_files = image_files[i:i+batch_size]
        batch_images = []
        batch_texts = []

        for j, file in enumerate(batch_files):
            img = Image.open(os.path.join(image_dir, file)).convert("RGB")
            batch_images.append(img)

            # match prompt index
            idx = i + j
            batch_texts.append(prompts[idx])

        inputs = processor(
            text=batch_texts,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        ).to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            # Normalize (CRITICAL)
            image_embeds = F.normalize(image_embeds, dim=-1)
            text_embeds = F.normalize(text_embeds, dim=-1)

            # cosine similarity
            batch_scores = (image_embeds * text_embeds).sum(dim=-1)

        scores.extend(batch_scores.cpu().numpy())

    return float(np.mean(scores))

def compute_multiple_fid_and_clip(real_dir, fake_dir_list):
    # Get all FID and CLIP scores
    fid_scores = []
    clip_scores = []
    model, processor = load_clip()

    for dir in fake_dir_list:
        fid_scores.append(compute_fid(real_dir, dir))
        clip_scores.append(compute_clip_score(dir, prompts, model, processor, 32))

    return fid_scores, clip_scores

### Get all generated images from google drive

In [ ]:
!unzip /content/drive/MyDrive/data/DDIM_15s_sd1.5_images.zip -d /
!unzip /content/drive/MyDrive/data/DDIM_sd1.5_images.zip -d /
!unzip /content/drive/MyDrive/data/base_sd1.5_images.zip -d /
!unzip /content/drive/MyDrive/data/DDIM_15s_sd2.1_images.zip -d /
!unzip /content/drive/MyDrive/data/DDIM_sd2.1_images.zip -d /
!unzip /content/drive/MyDrive/data/base_sd2.1_images.zip -d /

### Calculate FID and CLIP for each run

In [ ]:
generated_image_folders = ["/content/base_sd1.5_images",
                           "/content/DDIM_sd1.5_images",
                           "/content/DDIM_15s_sd1.5_images",
                           "/content/base_sd2.1_images",
                           "/content/DDIM_sd2.1_images",
                           "/content/DDIM_15s_sd2.1_images",]
print(compute_multiple_fid_and_clip("/content/coco_real", generated_image_folders))

### Create results table

In [2]:
import pandas as pd
import numpy as np

data = [{'exp_name': 'SD1.5 False',
 'total_time': 412.0782401561737,
 'throughput': 2.426723623215362,
 'avg_time_per_image': 0.4120782401561737,
 'std_per_image': np.float64(0.009731813032321403),
 'peak_vram_mb': 19053.4462890625},
 {'exp_name': 'SD1.5 True',
 'total_time': 397.6759307384491,
 'throughput': 2.5146103213817548,
 'avg_time_per_image': 0.3976759307384491,
 'std_per_image': np.float64(0.010165800870192674),
 'peak_vram_mb': 19050.4462890625},
  {'exp_name': 'SD1.5 True (15 steps)',
 'total_time': 263.4838283061981,
 'throughput': 3.7952993412479437,
 'avg_time_per_image': 0.2634838283061981,
 'std_per_image': np.float64(0.008706902415744023),
 'peak_vram_mb': 19050.4462890625},
  {'exp_name': 'SD2.1 False',
 'total_time': 851.6565268039703,
 'throughput': 1.1741822771589874,
 'avg_time_per_image': 0.8516565268039703,
 'std_per_image': np.float64(0.012436368021010759),
 'peak_vram_mb': 35929.74072265625},
  {'exp_name': 'SD2.1 True',
 'total_time': 862.4593341350555,
 'throughput': 1.1594749577414933,
 'avg_time_per_image': 0.8624593341350555,
 'std_per_image': np.float64(0.011597138041384155),
 'peak_vram_mb': 35929.74072265625},
 {'exp_name': 'SD2.1 True (15 steps)',
 'total_time': 547.1608603000641,
 'throughput': 1.8276161044333434,
 'avg_time_per_image': 0.5471608603000641,
 'std_per_image': np.float64(0.010942630359414295),
 'peak_vram_mb': 35930.36572265625}]

df = pd.DataFrame(data)
df = df.set_index("exp_name")

fid_scores = [52.5096651538185, 52.74946265849644, 53.970192951884314, 52.75625135588018, 52.75625135588018, 52.83020732459198]
clip_scores = [0.1454213410615921, 0.14574800431728363, 0.14649848639965057, 0.1456415355205536, 0.1456415355205536, 0.14663806557655334]

df["FID"] = fid_scores
df["CLIP"] = clip_scores

table = pd.DataFrame({
    "Avg Time Per Img (s)": df["avg_time_per_image"],
    "Std Time Per Img (s)": df["std_per_image"],
    "VRAM (MB)": df["peak_vram_mb"],
    "FID": df["FID"],
    "CLIP": df["CLIP"],
    "Throughput (img/s)": df["throughput"],
}).T  # transpose

table = table[[
    "SD1.5 False",
    "SD1.5 True",
    "SD1.5 True (15 steps)",
    "SD2.1 False",
    "SD2.1 True",
    "SD2.1 True (15 steps)",
]]

table = table.round(3)
table

exp_name,SD1.5 False,SD1.5 True,SD1.5 True (15 steps),SD2.1 False,SD2.1 True,SD2.1 True (15 steps)
Avg Time Per Img (s),0.412,0.398,0.263,0.852,0.862,0.547
Std Time Per Img (s),0.010,0.010,0.009,0.012,0.012,0.011
VRAM (MB),19053.446,19050.446,19050.446,35929.741,35929.741,35930.366
FID,52.510,52.749,53.970,52.756,52.756,52.830
CLIP,0.145,0.146,0.146,0.146,0.146,0.147
Throughput (img/s),2.427,2.515,3.795,1.174,1.159,1.828
